In [1]:
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error

In [2]:
# Specify input and output paths
output_path = '-- precise output path, where the model will be saved --'
input_path = '-- precise input_path, where the datasets are saved --'

In [3]:
# Open training and testing sets
test = pd.read_pickle(input_path + 'test.pkl')
train = pd.read_pickle(input_path + 'train.pkl')

In [4]:
# Input column names
lsfeat = ['AL', 'K', 'AQD', 'LT', 'CCT', 'WTW', 'Power']
# Target column name
target = 'ES_postop'

In [5]:
X_train = train[lsfeat].values
X_test = test[lsfeat].values
y_train = train[target].values
y_test = test[target].values

In [6]:
# DMatrix creation
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

In [7]:
# Initial parameters
params = {
    # Parameters that we are going to tune.
    'max_depth':6,
    'min_child_weight': 1,
    'eta':.3,
    'subsample': 1,
    'colsample_bytree': 1,
    # Other parameters
    'objective':'reg:squarederror'}

In [8]:
params['eval_metric'] = "mae"
num_boost_round = 999

In [9]:
model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")],
    early_stopping_rounds=10
)

[0]	Test-mae:0.31193
[1]	Test-mae:0.30553
[2]	Test-mae:0.30329
[3]	Test-mae:0.30055
[4]	Test-mae:0.30025
[5]	Test-mae:0.29960
[6]	Test-mae:0.29865
[7]	Test-mae:0.29749
[8]	Test-mae:0.29687
[9]	Test-mae:0.29682
[10]	Test-mae:0.29571
[11]	Test-mae:0.29742
[12]	Test-mae:0.29799
[13]	Test-mae:0.29755
[14]	Test-mae:0.29764
[15]	Test-mae:0.29746
[16]	Test-mae:0.29821
[17]	Test-mae:0.29888
[18]	Test-mae:0.29916
[19]	Test-mae:0.29901
[20]	Test-mae:0.29892


In [10]:
cv_results = xgb.cv(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    seed=99,
    nfold=5,
    metrics={'mae'},
    early_stopping_rounds=10
)

In [11]:
# Initial MAE
cv_results['test-mae-mean'].min()

0.2769848705379145

In [12]:
# max_depth and min_child_weight tuning
gridsearch_params = [
    (max_depth, min_child_weight)
    for max_depth in range(9,12)
    for min_child_weight in range(5,8)
]

In [13]:
min_mae = float("Inf")
best_params = None
for max_depth, min_child_weight in gridsearch_params:
    print("max_depth={}, min_child_weight={}".format(
                             max_depth,
                             min_child_weight))
    # Update parameters
    params['max_depth'] = max_depth
    params['min_child_weight'] = min_child_weight
    # Run CV
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        seed=42,
        nfold=5,
        metrics={'mae'},
        early_stopping_rounds=10
    )
    # Update best MAE
    mean_mae = cv_results['test-mae-mean'].min()
    boost_rounds = cv_results['test-mae-mean'].argmin()
    print("\tMAE {} after {} rounds".format(mean_mae, boost_rounds))
    if mean_mae < min_mae:
        min_mae = mean_mae
        best_params = (max_depth,min_child_weight)
print("Best params: {}, {}, MAE: {}".format(best_params[0], best_params[1], min_mae))

max_depth=9, min_child_weight=5
	MAE 0.27947667060074227 after 8 rounds
max_depth=9, min_child_weight=6
	MAE 0.2799371310884465 after 7 rounds
max_depth=9, min_child_weight=7
	MAE 0.2785184742985142 after 14 rounds
max_depth=10, min_child_weight=5
	MAE 0.2788281053148511 after 7 rounds
max_depth=10, min_child_weight=6
	MAE 0.2779150746924652 after 10 rounds
max_depth=10, min_child_weight=7
	MAE 0.2811219613872059 after 7 rounds
max_depth=11, min_child_weight=5
	MAE 0.28000944729172367 after 13 rounds
max_depth=11, min_child_weight=6
	MAE 0.2801360671898693 after 7 rounds
max_depth=11, min_child_weight=7
	MAE 0.2813907150910059 after 7 rounds
Best params: 10, 6, MAE: 0.2779150746924652


In [14]:
# best max_depth and min_child_weight are specified
params['max_depth'] = best_params[0]
params['min_child_weight'] = best_params[1]

In [15]:
# subsample and colsample tuning
gridsearch_params = [
    (subsample, colsample)
    for subsample in [i/10. for i in range(7,11)]
    for colsample in [i/10. for i in range(7,11)]
]

In [16]:
min_mae = float("Inf")
best_params = None

for subsample, colsample in reversed(gridsearch_params):
    print("subsample={}, colsample={}".format(
                             subsample,
                             colsample))
    # Update parameters
    params['subsample'] = subsample
    params['colsample_bytree'] = colsample
    # Run CV
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        seed=42,
        nfold=5,
        metrics={'mae'},
        early_stopping_rounds=10
    )
    # Update best score
    mean_mae = cv_results['test-mae-mean'].min()
    boost_rounds = cv_results['test-mae-mean'].argmin()
    print("\tMAE {} after {} rounds".format(mean_mae, boost_rounds))
    if mean_mae < min_mae:
        min_mae = mean_mae
        best_params = (subsample,colsample)
        
print("Best params: {}, {}, MAE: {}".format(best_params[0], best_params[1], min_mae))

subsample=1.0, colsample=1.0
	MAE 0.2779150746924652 after 10 rounds
subsample=1.0, colsample=0.9
	MAE 0.28081436304056395 after 21 rounds
subsample=1.0, colsample=0.8
	MAE 0.2825472690107981 after 9 rounds
subsample=1.0, colsample=0.7
	MAE 0.2812195047729897 after 13 rounds
subsample=0.9, colsample=1.0
	MAE 0.2810291575744096 after 12 rounds
subsample=0.9, colsample=0.9
	MAE 0.2833365836604059 after 7 rounds
subsample=0.9, colsample=0.8
	MAE 0.2801294619639865 after 13 rounds
subsample=0.9, colsample=0.7
	MAE 0.2847927993688183 after 8 rounds
subsample=0.8, colsample=1.0
	MAE 0.2804982506264424 after 9 rounds
subsample=0.8, colsample=0.9
	MAE 0.2819829572798976 after 7 rounds
subsample=0.8, colsample=0.8
	MAE 0.28284718507560824 after 8 rounds
subsample=0.8, colsample=0.7
	MAE 0.28614915147025366 after 10 rounds
subsample=0.7, colsample=1.0
	MAE 0.2834004982381938 after 6 rounds
subsample=0.7, colsample=0.9
	MAE 0.28465693304208683 after 5 rounds
subsample=0.7, colsample=0.8
	MAE 0.28

In [17]:
# best subsample and colsample values are specified
params['subsample'] = best_params[0]
params['colsample_bytree'] = best_params[1]

In [18]:
# eta tuning
min_mae = float("Inf")
best_params = None
for eta in [.3, .2, .1, .05, .01, .005]:
    print("eta={}".format(eta))
    # Update parameters
    params['eta'] = eta
    # Run and time CV
    cv_results = xgb.cv(
            params,
            dtrain,
            num_boost_round=num_boost_round,
            seed=42,
            nfold=5,
            metrics=['mae'],
            early_stopping_rounds=10
          )
    # Update best score
    mean_mae = cv_results['test-mae-mean'].min()
    boost_rounds = cv_results['test-mae-mean'].argmin()
    print("\tMAE {} after {} rounds\n".format(mean_mae, boost_rounds))
    if mean_mae < min_mae:
        min_mae = mean_mae
        best_params = eta
print("Best params: {}, MAE: {}".format(best_params, min_mae))

eta=0.3
	MAE 0.2779150746924652 after 10 rounds

eta=0.2
	MAE 0.2770873749589329 after 36 rounds

eta=0.1
	MAE 0.27073432770559586 after 79 rounds

eta=0.05
	MAE 0.27066466591037786 after 157 rounds

eta=0.01
	MAE 0.27134014415520236 after 391 rounds

eta=0.005
	MAE 0.2714287396068581 after 627 rounds

Best params: 0.05, MAE: 0.27066466591037786


In [19]:
# best eta is specified
params['eta'] = best_params

In [20]:
model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")],
    early_stopping_rounds=10
)

print("Best MAE: {:.2f} in {} rounds".format(model.best_score, model.best_iteration+1))

[0]	Test-mae:0.32006
[1]	Test-mae:0.31854
[2]	Test-mae:0.31687
[3]	Test-mae:0.31524
[4]	Test-mae:0.31383
[5]	Test-mae:0.31212
[6]	Test-mae:0.31074
[7]	Test-mae:0.30987
[8]	Test-mae:0.30863
[9]	Test-mae:0.30774
[10]	Test-mae:0.30717
[11]	Test-mae:0.30686
[12]	Test-mae:0.30608
[13]	Test-mae:0.30575
[14]	Test-mae:0.30497
[15]	Test-mae:0.30450
[16]	Test-mae:0.30435
[17]	Test-mae:0.30394
[18]	Test-mae:0.30319
[19]	Test-mae:0.30295
[20]	Test-mae:0.30294
[21]	Test-mae:0.30286
[22]	Test-mae:0.30232
[23]	Test-mae:0.30194
[24]	Test-mae:0.30152
[25]	Test-mae:0.30094
[26]	Test-mae:0.30073
[27]	Test-mae:0.30015
[28]	Test-mae:0.29958
[29]	Test-mae:0.29915
[30]	Test-mae:0.29918
[31]	Test-mae:0.29884
[32]	Test-mae:0.29868
[33]	Test-mae:0.29870
[34]	Test-mae:0.29872
[35]	Test-mae:0.29901
[36]	Test-mae:0.29873
[37]	Test-mae:0.29879
[38]	Test-mae:0.29884
[39]	Test-mae:0.29867
[40]	Test-mae:0.29861
[41]	Test-mae:0.29852
[42]	Test-mae:0.29872
[43]	Test-mae:0.29882
[44]	Test-mae:0.29923
[45]	Test-mae:0.2989

In [21]:
num_boost_round = model.best_iteration + 1
best_model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")]
)

[0]	Test-mae:0.32006
[1]	Test-mae:0.31854
[2]	Test-mae:0.31687
[3]	Test-mae:0.31524
[4]	Test-mae:0.31383
[5]	Test-mae:0.31212
[6]	Test-mae:0.31074
[7]	Test-mae:0.30987
[8]	Test-mae:0.30863
[9]	Test-mae:0.30774
[10]	Test-mae:0.30717
[11]	Test-mae:0.30686
[12]	Test-mae:0.30608
[13]	Test-mae:0.30575
[14]	Test-mae:0.30497
[15]	Test-mae:0.30450
[16]	Test-mae:0.30435
[17]	Test-mae:0.30394
[18]	Test-mae:0.30319
[19]	Test-mae:0.30295
[20]	Test-mae:0.30294
[21]	Test-mae:0.30286
[22]	Test-mae:0.30232
[23]	Test-mae:0.30194
[24]	Test-mae:0.30152
[25]	Test-mae:0.30094
[26]	Test-mae:0.30073
[27]	Test-mae:0.30015
[28]	Test-mae:0.29958
[29]	Test-mae:0.29915
[30]	Test-mae:0.29918
[31]	Test-mae:0.29884
[32]	Test-mae:0.29868
[33]	Test-mae:0.29870
[34]	Test-mae:0.29872
[35]	Test-mae:0.29901
[36]	Test-mae:0.29873
[37]	Test-mae:0.29879
[38]	Test-mae:0.29884
[39]	Test-mae:0.29867
[40]	Test-mae:0.29861
[41]	Test-mae:0.29852


In [22]:
mean_absolute_error(best_model.predict(dtest), y_test)

0.29852117347720414

In [23]:
best_model.save_model(output_path + "AItoSE.model")